<a href="https://colab.research.google.com/github/GarretOS/python-ai-foundations/blob/main/projects/nasa-space-rag/nasa_space_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🚀 NASA Space RAG

This notebook teaches and demonstrates a beginner-friendly retrieval-augmented generation application based on the TowardsAI **Chat with Your Documents: A RAG Application** lesson. It uses Gemini and Gradio to answer questions from five NASA-derived text documents.

## 🎯 Project Overview

The application embeds the documents, retrieves the two documents most similar to a question, and asks Gemini to answer using only their text. The Gradio interface displays the answer, the retrieved document names, and their actual cosine similarity scores. The local `nasa_space_rag.py` file is the primary application; this notebook is a self-contained Colab learning companion.

## 🐍 Python Concepts

This project practices:

- Classes, objects, and methods
- Functions, parameters, and return values
- Lists and dictionaries
- JSON file reading and writing
- Manual cosine similarity with arithmetic, `sum()`, `zip()`, and `math.sqrt()`
- Sorting documents by their similarity scores
- Calling Gemini and connecting a Python method to Gradio components

## 🧩 How the RAG Workflow Works

RAG means **retrieval-augmented generation**. This application first finds relevant information in its document collection, then gives that information to Gemini as context for an answer.

1. Load five NASA text documents.
2. Turn each document into a numeric embedding and save the embeddings in a JSON cache.
3. Turn a question into an embedding.
4. Compare the question with every document using cosine similarity, then retrieve the two highest-scoring documents.
5. Ask Gemini to answer using only those two documents.

The model does not search the web. Its answer is grounded in the text included in the prompt.

## 🏗️ Classes and Objects

A **class** groups related data and functions. `SpaceAssistant` keeps the documents, embeddings, and Gemini client together. Its methods load documents, create and reuse embeddings, compare vectors, retrieve documents, and answer questions.

An **object** is one working instance of a class. Later, `assistant = SpaceAssistant(api_key)` creates the object that the Gradio interface calls.

## 📦 Install the Dependencies

`google-genai` connects to Gemini, and `gradio` builds the interactive interface. Colab installs them into the current runtime.

In [ ]:
%pip install -q google-genai gradio

## 🔐 Set Up Your Gemini API Key Safely

In Colab, open the **Secrets** panel (key icon), add a secret named `GEMINI_API_KEY`, and allow this notebook to access it. The next cell reads the secret without displaying its value. Never paste a real key into a notebook cell or commit it to GitHub.

In [ ]:
from google.colab import userdata

api_key = userdata.get("GEMINI_API_KEY")
if not api_key:
    raise ValueError("Add GEMINI_API_KEY to Colab Secrets and enable notebook access.")

print("Gemini API key loaded from Colab Secrets.")

## 📦 Import Modules

`json`, `math`, `pathlib`, and `urllib` are part of Python. `google-genai` provides the Gemini client, and `gradio` builds the interface.

In [ ]:
import json
import math
from pathlib import Path
from urllib.request import urlopen

import gradio as gr
from google import genai

EMBEDDING_MODEL = "gemini-embedding-2"
GENERATION_MODEL = "gemini-3.5-flash-lite"
CACHE_PATH = Path("doc_embeddings.json")
NO_ANSWER = "I can't find the answer in the NASA documents."

## 📄 Get the NASA Reference Documents

A new Colab session does not contain this repository's files. The next cell downloads the five text files from this project's public GitHub repository using Python's built-in `urllib` library. No NASA pages are scraped.

## 🌌 NASA Document Sources

The text files are derived from these NASA reference pages:

- [NASA Science — Mars Facts](https://science.nasa.gov/mars/facts/)
- [NASA Science — Moon Phases](https://science.nasa.gov/moon/moon-phases/)
- [NASA Science — Solar System Facts](https://science.nasa.gov/solar-system/solar-system-facts/)
- [NASA — Space Station Research and Technology](https://www.nasa.gov/international-space-station/space-station-research-and-technology/)
- [NASA Science — Black Holes](https://science.nasa.gov/universe/black-holes/)

In [ ]:
RAW_DOCUMENT_URLS = {
    "black_holes.txt": "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/main/projects/nasa-space-rag/documents/black_holes.txt",
    "international_space_station.txt": "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/main/projects/nasa-space-rag/documents/international_space_station.txt",
    "mars.txt": "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/main/projects/nasa-space-rag/documents/mars.txt",
    "moon.txt": "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/main/projects/nasa-space-rag/documents/moon.txt",
    "solar_system.txt": "https://raw.githubusercontent.com/GarretOS/python-ai-foundations/main/projects/nasa-space-rag/documents/solar_system.txt",
}

DOCUMENTS_FOLDER = Path("documents")
DOCUMENTS_FOLDER.mkdir(exist_ok=True)

for filename, url in RAW_DOCUMENT_URLS.items():
    with urlopen(url) as response:
        document_text = response.read().decode("utf-8")
    (DOCUMENTS_FOLDER / filename).write_text(document_text, encoding="utf-8")

print(f"Downloaded {len(RAW_DOCUMENT_URLS)} NASA reference documents.")
print("\n".join(sorted(path.name for path in DOCUMENTS_FOLDER.glob("*.txt"))))

## 🧠 Embeddings and the JSON Cache

An **embedding** represents text as a list of numbers. Texts with related meaning often have embeddings that point in similar directions. `gemini-embedding-2` creates one embedding for each NASA document and for each question.

Embedding the five documents uses API requests, so the application saves the documents and their aligned embeddings in `doc_embeddings.json`. Later runs reuse the cache when it is valid and records the same embedding model. In Colab, the cache is stored in the current runtime and may disappear when that runtime is reset.

## 🧮 Manual Cosine Similarity

Cosine similarity compares the direction of two vectors. Here it measures how closely a question embedding aligns with each document embedding. The method below uses Python arithmetic, `sum()`, `zip()`, and `math.sqrt()`; it does not use a vector-search framework. A higher score means a closer direction, but it does not guarantee that the document contains a complete answer.

## 🔎 Retrieval and Grounded Answer Generation

The retrieval method scores every document, sorts by the actual cosine similarity, and returns the top two. The answer method includes those two text passages in a prompt that tells Gemini to use only the supplied NASA documents. If the documents do not support an answer, the model is instructed to return: `I can't find the answer in the NASA documents.`

In [ ]:
class SpaceAssistant:
    """Manage NASA documents, embeddings, retrieval, and answers."""

    def __init__(self, api_key):
        self.documents = []
        self.embeddings = []
        self.client = genai.Client(api_key=api_key)

    def load_documents(self, documents_folder):
        """Load text documents from a folder in filename order."""
        documents_folder = Path(documents_folder)
        document_paths = sorted(documents_folder.glob("*.txt"))
        if not document_paths:
            raise FileNotFoundError(f"No .txt documents found in {documents_folder}")

        self.documents = [
            {"filename": path.name, "content": path.read_text(encoding="utf-8")}
            for path in document_paths
        ]

    def embed_documents(self):
        """Create an embedding for every loaded document."""
        if not self.documents:
            raise ValueError("Load documents before creating embeddings.")

        self.embeddings = []
        for document in self.documents:
            response = self.client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=document["content"],
            )
            self.embeddings.append(response.embeddings[0].values)

    def save_embeddings(self, cache_path):
        """Save documents and their matching embeddings as JSON."""
        if len(self.documents) != len(self.embeddings):
            raise ValueError("Documents and embeddings must have matching lengths.")

        cache = {
            "documents": self.documents,
            "embeddings": self.embeddings,
            "embedding_model": EMBEDDING_MODEL,
        }
        with open(cache_path, "w", encoding="utf-8") as cache_file:
            json.dump(cache, cache_file)

    def load_embeddings(self, cache_path):
        """Reuse a readable cache when it matches this embedding model."""
        try:
            with open(cache_path, "r", encoding="utf-8") as cache_file:
                cache = json.load(cache_file)

            documents = cache["documents"]
            embeddings = cache["embeddings"]
            if not isinstance(documents, list) or not isinstance(embeddings, list):
                raise ValueError("Cache documents and embeddings must be lists.")
            if len(documents) != len(embeddings) or not documents:
                raise ValueError("Cache documents and embeddings are not aligned.")
            if cache.get("embedding_model") not in (None, EMBEDDING_MODEL):
                raise ValueError("Cache uses a different embedding model.")

            self.documents = documents
            self.embeddings = embeddings
            return True
        except (OSError, json.JSONDecodeError, KeyError, TypeError, ValueError) as error:
            print(f"Could not load embeddings cache: {error}")
            return False

    def cosine_similarity(self, vector_a, vector_b):
        """Calculate cosine similarity with basic Python operations."""
        if len(vector_a) != len(vector_b):
            raise ValueError("Vectors must have the same length.")

        dot_product = sum(a * b for a, b in zip(vector_a, vector_b))
        magnitude_a = math.sqrt(sum(value * value for value in vector_a))
        magnitude_b = math.sqrt(sum(value * value for value in vector_b))
        if magnitude_a == 0 or magnitude_b == 0:
            return 0.0
        return dot_product / (magnitude_a * magnitude_b)

    def retrieve_documents(self, question_embedding, top_k=2):
        """Return the top-scoring documents with their similarity scores."""
        if len(self.documents) != len(self.embeddings):
            raise ValueError("Documents and embeddings are not aligned.")

        scored_documents = []
        for index, embedding in enumerate(self.embeddings):
            score = self.cosine_similarity(question_embedding, embedding)
            scored_documents.append({
                "filename": self.documents[index]["filename"],
                "content": self.documents[index]["content"],
                "score": score,
            })

        scored_documents.sort(key=lambda document: document["score"], reverse=True)
        return scored_documents[:top_k]

    def answer_question(self, question):
        """Retrieve context and ask Gemini for a grounded answer."""
        if not question or not question.strip():
            return "Please enter a question.", "No documents retrieved."
        if not self.documents:
            return "No NASA documents are loaded.", "No documents retrieved."
        if len(self.documents) != len(self.embeddings):
            return (
                "Embeddings are unavailable. Load a valid cache or create embeddings first.",
                "No documents retrieved.",
            )

        try:
            question_response = self.client.models.embed_content(
                model=EMBEDDING_MODEL,
                contents=question.strip(),
            )
            question_embedding = question_response.embeddings[0].values
            retrieved_documents = self.retrieve_documents(question_embedding)
            context = "\n\n".join(
                f"Document: {document['filename']}\n{document['content']}"
                for document in retrieved_documents
            )
            prompt = f"""Answer the question using only the NASA documents below.
If the documents do not support an answer, say exactly: "{NO_ANSWER}"
Do not use outside knowledge.

Question: {question.strip()}

NASA documents:
{context}
"""
            answer_response = self.client.models.generate_content(
                model=GENERATION_MODEL,
                contents=prompt,
            )
            answer = answer_response.text or NO_ANSWER
            retrieval_text = "\n".join(
                f"{document['filename']} — {document['score']:.3f}"
                for document in retrieved_documents
            )
            return answer, retrieval_text
        except Exception as error:
            return f"Gemini API error: {error}", "Retrieved documents were not sent as an answer."

## ⚙️ Load the Documents and Reuse the Cache

The cell below creates the assistant and loads the downloaded documents. If a valid JSON cache already exists, it reuses the saved embeddings. Otherwise it asks Gemini to create document embeddings and saves them for later in this Colab runtime. Document embedding requests may use your API quota.

In [ ]:
assistant = SpaceAssistant(api_key)
assistant.load_documents(DOCUMENTS_FOLDER)

if assistant.load_embeddings(CACHE_PATH):
    print(f"Loaded {len(assistant.documents)} documents and embeddings from cache.")
else:
    assistant.embed_documents()
    assistant.save_embeddings(CACHE_PATH)
    print(f"Created and cached embeddings for {len(assistant.documents)} documents.")

print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Generation model: {GENERATION_MODEL}")

## 🧪 Example Questions and Expected Behavior

Try these questions in the interface:

- `What are the phases of the Moon?` — should retrieve `moon.txt` near the top and answer from its text.
- `What is the International Space Station used for?` — should use `international_space_station.txt` as supporting context.
- `What is a black hole?` — should use `black_holes.txt` as supporting context.
- `What is my favorite color?` — the NASA documents do not provide this personal information, so the expected fallback is `I can't find the answer in the NASA documents.`

The exact ranking and wording can vary. For each valid question, the second output shows the two retrieved filenames and their actual cosine similarity scores, rounded to three decimal places. Similarity is a retrieval measurement, not a model confidence score.

## 🖥️ Build and Launch the Gradio Interface

The Gradio button calls `assistant.answer_question()`. The interface displays the generated answer and a separate list of the top two retrieved filenames with their similarity scores. `share=True` creates a temporary public Gradio link, which allows the interface to open from a Colab session. Anyone with that temporary link may be able to access the demo while it is running, so stop the cell when you are finished.

In [ ]:
with gr.Blocks(title="NASA Space RAG") as demo:
    gr.Markdown("# 🚀 NASA Space RAG\nAsk a question about the five NASA reference documents.")
    question = gr.Textbox(
        label="Question",
        placeholder="Ask a question about NASA space topics...",
    )
    ask_button = gr.Button("Ask")
    answer = gr.Textbox(label="Answer", lines=6)
    retrieved = gr.Textbox(
        label="Top two retrieved documents and cosine similarity",
        lines=3,
    )
    ask_button.click(
        fn=assistant.answer_question,
        inputs=question,
        outputs=[answer, retrieved],
    )

demo.launch(share=True)

## 📚 What I Learned

This project connects a Python class and objects with Gemini embeddings, a JSON cache, manual cosine similarity, document retrieval, and grounded text generation. I also practiced retrieving a secret safely in Colab and connecting a Python method to a Gradio interface.

## 📝 Notes

- Gemini API access, quotas, model availability, and an internet connection are required.
- The notebook downloads the five committed `.txt` files from this public GitHub repository.
- The embedding cache is stored in the Colab runtime and is not included in the project repository.
- The application uses whole documents and does not split them into smaller chunks.
- Answers are limited to information supported by the five included NASA-derived text files. Retrieval scores help identify related text but do not guarantee a complete answer.
- Never save a real API key in a notebook cell or share it in a screenshot.